# Laguna XS.2 — Causal Expert Atlas on 1× L40S 48GB
## v9 — quantized vLLM backend, single GPU, causal-first

This notebook replaces the Transformers backend for the **atlas/localization** stage.

### Why

Current Transformers converts Laguna's packed per-expert MoE weights into fused
expert tensors. For these fused MoE tensors, its compressed-tensors integration
uses a `DecompressExperts` conversion path, which materializes experts in BF16.
For XS.2, the routed experts alone are about **58.5 GiB in BF16**, so a 44.4 GiB
L40S cannot hold the converted model.

vLLM instead has a native fused-MoE quantized execution path for Laguna and can
keep the mixed INT4 / group-INT8 expert checkpoint compressed.

### Hardware target

- 1× NVIDIA L40S (48 GB advertised / ~44.4 GiB)
- 4 CPU cores
- 32 GB RAM
- local checkpoint already present at:
  `/home/ec2-user/workspace/models/Laguna-XS.2-INT4`

Override with environment variable `LAGUNA_MODEL_PATH` if needed.

### What this notebook does

1. verifies the local checkpoint and L40S;
2. installs/uses vLLM 0.27.1;
3. applies upstream grouped-INT8 compatibility fix #47154 if this wheel still needs it;
4. instruments the actual MoE runner **after real top-k routing**;
5. measures target/control reference-token NLL;
6. runs causal layer ablation;
7. runs hierarchical expert-group search;
8. validates leaf experts individually;
9. checks renormalized interventions;
10. compares causal rank against routing frequency only afterward.

### Intervention semantics

For expert `e`, the router still selects the original top-k IDs.

We then set only the routing weight of `e` to zero:

```text
original top-8 IDs  -> unchanged
weight(e)           -> 0
expert #9           -> NOT substituted
```

This is the fixed-routing intervention we want.

## 1 — Install runtime in a fresh kernel

In [ ]:
# Run this before importing vLLM.
# If your environment already has 0.27.1, pip will be quick.
%pip -q install -U "vllm==0.27.1" "transformers>=5.14.1" pandas numpy psutil pyarrow

## 2 — Low-CPU / single-GPU environment

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MALLOC_ARENA_MAX"] = "2"

# Avoid an irrelevant FlashInfer sampler warning on some installations.
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

print("Environment configured.")

## 3 — Hardware + checkpoint preflight

In [ ]:
import os, json, shutil
from pathlib import Path
import psutil
import torch

MODEL_PATH = Path(
    os.environ.get(
        "LAGUNA_MODEL_PATH",
        "/home/ec2-user/workspace/models/Laguna-XS.2-INT4",
    )
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00005.safetensors"
    for i in range(1, 6)
]

print("=== Host ===")
ram = psutil.virtual_memory()
print("CPU logical cores:", os.cpu_count())
print(f"RAM total: {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f"This notebook expects exactly one GPU; found {torch.cuda.device_count()}."
    )

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory < 47_000_000_000:
    raise RuntimeError("Need a 48GB-class GPU for this notebook.")

if torch.cuda.get_device_capability(0) < (8, 9):
    print(
        "WARNING: this notebook is tuned for L40S/Ada SM89. "
        "FP8 KV behavior may differ on older GPUs."
    )

print("\n=== Model ===")
print("MODEL_PATH:", MODEL_PATH)

if not MODEL_PATH.exists():
    raise RuntimeError(
        "Local checkpoint not found. Set LAGUNA_MODEL_PATH to the existing model directory."
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]
if missing:
    raise RuntimeError(f"Checkpoint is incomplete; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]
for name, n in sizes:
    print(f"{name}: {n/1e9:.3f} GB")
print(f"Weight shards total: {sum(n for _, n in sizes)/1e9:.3f} GB")

print("\nPreflight: PASS")

## 4 — Patch vLLM before importing it

Two narrowly-scoped patches are applied.

### A. Grouped INT8 WNA16 compatibility

Poolside's XS.2-INT4 checkpoint mixes INT4 experts with group-quantized INT8
experts. vLLM upstream PR #47154 removed an obsolete:

```python
assert self.group_size == -1
```

Some packaged builds have still contained that assertion. We remove only that
exact line if present.

### B. Causal hook

We insert a small hook immediately after:

```python
topk_weights, topk_ids = self.router.select_experts(...)
```

The hook reads one tiny JSON control file and can:

- zero all routed-expert weights in a layer;
- zero selected expert IDs while preserving top-k IDs;
- optionally renormalize surviving weights;
- record aggregate routing counts/mass.

No expert weights are unpacked or modified.

In [ ]:
import importlib.util
from pathlib import Path
import shutil

spec = importlib.util.find_spec("vllm")
if spec is None or not spec.submodule_search_locations:
    raise RuntimeError("vLLM installation not found.")

vllm_root = Path(list(spec.submodule_search_locations)[0])
print("vLLM root:", vllm_root)

# ------------------------------------------------------------
# A. Exact upstream #47154 compatibility fix
# ------------------------------------------------------------
ct_dir = (
    vllm_root
    / "model_executor/layers/quantization/compressed_tensors"
    / "compressed_tensors_moe"
)

compat_files = [
    ct_dir / "compressed_tensors_moe_wna16.py",
    ct_dir / "compressed_tensors_moe_wna16_marlin.py",
]

found = False
for p in compat_files:
    if not p.exists():
        continue
    found = True
    txt = p.read_text()
    obsolete = "        assert self.group_size == -1\n"

    if obsolete in txt:
        bak = p.with_suffix(p.suffix + ".pre_laguna_pr47154")
        if not bak.exists():
            shutil.copy2(p, bak)
        p.write_text(txt.replace(obsolete, "", 1))
        print("Applied grouped-INT8 fix:", p)
    else:
        print("Grouped-INT8 assertion already absent:", p)

    if "assert self.group_size == -1" in p.read_text():
        raise RuntimeError(f"Obsolete grouped-INT8 assertion remains in {p}")

if not found:
    raise RuntimeError("Could not locate vLLM compressed-tensors MoE implementation.")

# ------------------------------------------------------------
# B. Fixed-routing causal hook
# ------------------------------------------------------------
runner_path = (
    vllm_root
    / "model_executor/layers/fused_moe/runner/moe_runner.py"
)

if not runner_path.exists():
    raise RuntimeError(f"MoE runner not found: {runner_path}")

src = runner_path.read_text()
PATCH_MARK = "# === LAGUNA_L40S_CAUSAL_PATCH_V1 ==="

if PATCH_MARK not in src:
    bak = runner_path.with_suffix(".py.pre_laguna_l40s_causal")
    if not bak.exists():
        shutil.copy2(runner_path, bak)

    import_anchor = "from collections.abc import Callable, Iterable\n"
    if import_anchor not in src:
        raise RuntimeError(
            "Unexpected vLLM moe_runner.py import layout. "
            "Do not apply an unsafe patch."
        )

    src = src.replace(
        import_anchor,
        import_anchor
        + "import json as _laguna_causal_json\n"
        + "import os as _laguna_causal_os\n",
        1,
    )

    select_block = '''            topk_weights, topk_ids = self.router.select_experts(
                hidden_states=hidden_states,
                router_logits=router_logits,
                topk_indices_dtype=self._quant_method.topk_indices_dtype,
                input_ids=input_ids,
            )
'''

    if select_block not in src:
        idx = src.find("topk_weights, topk_ids")
        if idx >= 0:
            print(src[max(0, idx-600):idx+1400])
        raise RuntimeError(
            "Expected vLLM select_experts block was not found. "
            "Stop rather than patching the wrong code."
        )

    hook = r'''
            # === LAGUNA_L40S_CAUSAL_PATCH_V1 ===
            _ctl_path = _laguna_causal_os.environ.get("LAGUNA_CAUSAL_CONTROL", "")
            if _ctl_path and _laguna_causal_os.path.exists(_ctl_path):
                try:
                    with open(_ctl_path, "r") as _f:
                        _ctl = _laguna_causal_json.load(_f)
                except Exception:
                    _ctl = {}

                _layer_name = str(getattr(self, "layer_name", ""))
                _zero_layers = set(_ctl.get("zero_layers", []))
                _targets = _ctl.get("ablate", {}).get(_layer_name, [])

                # Capture ORIGINAL routing before intervention.
                if bool(_ctl.get("capture", False)):
                    try:
                        _ids = topk_ids.detach().reshape(-1).to(torch.int64)
                        _ws = topk_weights.detach().reshape(-1).float()
                        _valid = _ids >= 0
                        _ids = _ids[_valid]
                        _ws = _ws[_valid]

                        if _ids.numel() > 0:
                            _n = int(self.moe_config.num_logical_experts)
                            _counts = torch.bincount(_ids, minlength=_n)
                            _wsum = torch.zeros(
                                _n,
                                device=_ws.device,
                                dtype=torch.float32,
                            )
                            _wsum.scatter_add_(0, _ids, _ws)
                            _active = torch.nonzero(
                                _counts > 0, as_tuple=False
                            ).reshape(-1)

                            _rec = {
                                "layer_name": _layer_name,
                                "tokens": int(hidden_states.shape[0]),
                                "counts": {
                                    str(int(i)): int(_counts[i].item())
                                    for i in _active
                                },
                                "weight_sums": {
                                    str(int(i)): float(_wsum[i].item())
                                    for i in _active
                                },
                            }

                            _base = _ctl.get("output_base")
                            if _base:
                                _out = str(_base) + ".jsonl"
                                with open(_out, "a") as _f:
                                    _f.write(_laguna_causal_json.dumps(_rec) + "\n")
                    except Exception:
                        # Instrumentation should never kill inference.
                        pass

                # Fixed top-k intervention.
                if _layer_name in _zero_layers:
                    topk_weights = torch.zeros_like(topk_weights)

                elif _targets:
                    _target_ids = torch.tensor(
                        _targets,
                        device=topk_ids.device,
                        dtype=topk_ids.dtype,
                    )
                    _keep = ~torch.isin(topk_ids, _target_ids)
                    topk_weights = topk_weights * _keep.to(topk_weights.dtype)

                    if bool(_ctl.get("renormalize", False)):
                        _den = topk_weights.sum(dim=-1, keepdim=True)
                        topk_weights = torch.where(
                            _den > 0,
                            topk_weights / _den.clamp_min(1e-12),
                            topk_weights,
                        )
            # === END LAGUNA_L40S_CAUSAL_PATCH_V1 ===
'''

    src = src.replace(select_block, select_block + hook, 1)
    runner_path.write_text(src)
    print("Inserted causal MoE hook.")
else:
    print("Causal MoE hook already present.")

verify = runner_path.read_text()
assert PATCH_MARK in verify
print("vLLM source preflight: PASS")

## 5 — Create control file and load XS.2

Because the intervention can change between every forward, we use
`enforce_eager=True`; CUDA graphs / compilation are intentionally disabled for
this research run.

L40S is SM89, so we leave Laguna's native FP8 KV cache enabled.

Important optimization choices:

- one GPU, no tensor parallelism;
- local checkpoint path, no worker download;
- BF16 compute for unquantized tensors;
- 1024 max context for short causal probes;
- up to 32 sequences so the whole evaluation set can be scheduled together;
- prefix caching disabled because cached hidden states would invalidate model interventions.

In [ ]:
import os, json, time
from pathlib import Path

WORK = Path("/home/ec2-user/workspace/laguna_causal_l40s")
WORK.mkdir(parents=True, exist_ok=True)

CONTROL_PATH = WORK / "control.json"
os.environ["LAGUNA_CAUSAL_CONTROL"] = str(CONTROL_PATH)

def write_control(*, capture=False, output_base=None, zero_layers=None,
                  ablate=None, renormalize=False):
    payload = {
        "capture": bool(capture),
        "output_base": str(output_base) if output_base is not None else None,
        "zero_layers": list(zero_layers or []),
        "ablate": {
            str(k): [int(x) for x in v]
            for k, v in (ablate or {}).items()
        },
        "renormalize": bool(renormalize),
    }
    tmp = CONTROL_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload))
    os.replace(tmp, CONTROL_PATH)

def clear_control():
    write_control()

clear_control()

from vllm import LLM, SamplingParams
from vllm.inputs import TokensPrompt
import vllm

print("vLLM:", vllm.__version__)

if vllm.__version__ != "0.27.1":
    raise RuntimeError(
        f"This notebook is validated for vLLM 0.27.1; found {vllm.__version__}."
    )

t0 = time.time()

llm = LLM(
    model=str(MODEL_PATH),
    trust_remote_code=True,

    dtype="bfloat16",
    max_model_len=1024,
    max_num_batched_tokens=4096,
    max_num_seqs=32,

    gpu_memory_utilization=0.90,
    cpu_offload_gb=0,

    enforce_eager=True,
    enable_prefix_caching=False,

    seed=42,
)

print(f"XS.2 loaded in {(time.time()-t0)/60:.2f} min")

## 6 — Generation smoke test

In [ ]:
out = llm.generate(
    ["Reply with exactly: atlas ready"],
    SamplingParams(temperature=0.0, max_tokens=8),
    use_tqdm=False,
)
print(repr(out[0].outputs[0].text))

## 7 — Router-capture smoke test

In [ ]:
import glob, json, re
from pathlib import Path

def remove_capture(base):
    for p in glob.glob(str(base) + "*.jsonl"):
        os.remove(p)

def read_capture(base):
    p = Path(str(base) + ".jsonl")
    if not p.exists():
        return []
    return [
        json.loads(line)
        for line in p.read_text().splitlines()
        if line.strip()
    ]

capture_base = WORK / "router_smoke"
remove_capture(capture_base)

write_control(
    capture=True,
    output_base=capture_base,
)

_ = llm.generate(
    ["A CSS flex child overflows its parent. Name one property you would inspect."],
    SamplingParams(temperature=0.0, max_tokens=1),
    use_tqdm=False,
)

clear_control()

router_smoke = read_capture(capture_base)

print("Capture records:", len(router_smoke))

MOE_LAYERS = sorted(
    set(r["layer_name"] for r in router_smoke),
    key=lambda s: int(re.search(r"layers\.(\d+)", s).group(1)),
)

print("Distinct MoE layers:", len(MOE_LAYERS))
print("First:", MOE_LAYERS[:3])
print("Last: ", MOE_LAYERS[-3:])

if len(MOE_LAYERS) < 35:
    raise RuntimeError(
        f"Expected roughly 39 sparse layers, captured {len(MOE_LAYERS)}. "
        "Stop before causal testing."
    )

print("Router hook: PASS")

# Phase A — Build tokenized target/control scoring prompts

The full prefix+reference is tokenized **once**.

Each causal intervention then sends one batch of already-tokenized prompts to
vLLM and asks for prompt log-probabilities. This minimizes CPU/tokenizer work on
the 4-core host.

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

EVAL_ROWS = [
    # ---------- target: frontend / UI ----------
    {"kind":"target","q":"In CSS flexbox, what declaration lets a flex item shrink below its intrinsic content width?","a":"min-width: 0;"},
    {"kind":"target","q":"What CSS declaration establishes a flex formatting context?","a":"display: flex;"},
    {"kind":"target","q":"Which React hook stores local component state?","a":"useState"},
    {"kind":"target","q":"What CSS property controls horizontal overflow?","a":"overflow-x"},
    {"kind":"target","q":"Which CSS property controls stacking order for positioned elements?","a":"z-index"},
    {"kind":"target","q":"Which declaration commonly centers flex children along the main axis?","a":"justify-content: center;"},
    {"kind":"target","q":"In React, which prop gives a stable identity to list items?","a":"key"},
    {"kind":"target","q":"Which CSS property sets spacing between grid or flex children without margins?","a":"gap"},
    {"kind":"target","q":"Which CSS property includes padding and border inside declared width?","a":"box-sizing"},
    {"kind":"target","q":"Which browser API observes element size changes?","a":"ResizeObserver"},
    {"kind":"target","q":"Which React hook runs side effects after rendering?","a":"useEffect"},
    {"kind":"target","q":"Which CSS property defines grid columns?","a":"grid-template-columns"},

    # ---------- controls ----------
    {"kind":"control","q":"Which Python keyword yields a value from a generator?","a":"yield"},
    {"kind":"control","q":"Which traversal finds shortest paths in an unweighted graph?","a":"BFS"},
    {"kind":"control","q":"Which Java keyword declares class inheritance?","a":"extends"},
    {"kind":"control","q":"Which SQL keyword removes duplicate SELECT rows?","a":"DISTINCT"},
    {"kind":"control","q":"Which C++ smart pointer represents exclusive ownership?","a":"std::unique_ptr"},
    {"kind":"control","q":"Which asymptotic notation describes an upper bound?","a":"Big O"},
    {"kind":"control","q":"Which Python container provides average O(1) membership lookup for hashable values?","a":"set"},
    {"kind":"control","q":"Which data structure is first-in first-out?","a":"queue"},
    {"kind":"control","q":"Which SQL clause filters groups after aggregation?","a":"HAVING"},
    {"kind":"control","q":"Which Java interface defines natural ordering?","a":"Comparable"},
    {"kind":"control","q":"Which C++ keyword prevents modification through that name?","a":"const"},
    {"kind":"control","q":"What mathematical operation is the inverse of exponentiation for solving an exponent?","a":"logarithm"},
]

eval_df = pd.DataFrame(EVAL_ROWS)

def make_prefix(question):
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

def tokenize_case(question, answer):
    prefix = make_prefix(question)
    full = prefix + " " + answer

    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    full_ids = tokenizer.encode(full, add_special_tokens=False)

    # Robust to one tokenizer merge at prefix/answer boundary.
    lcp = 0
    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        lcp += 1

    return {
        "prefix": prefix,
        "full": full,
        "prompt_token_ids": full_ids,
        "answer_start": lcp,
    }

cases = [
    tokenize_case(r.q, r.a)
    for r in eval_df.itertuples(index=False)
]

TOKEN_PROMPTS = [
    TokensPrompt(prompt_token_ids=c["prompt_token_ids"])
    for c in cases
]

print(eval_df.groupby("kind").size())
print("Max tokenized length:", max(len(c["prompt_token_ids"]) for c in cases))

## 8 — Batched teacher-forced reference NLL

In [ ]:
def _lp_value(x):
    if x is None:
        return None
    if isinstance(x, (float, int)):
        return float(x)
    if hasattr(x, "logprob"):
        return float(x.logprob)
    if isinstance(x, dict) and "logprob" in x:
        return float(x["logprob"])
    return None

def _chosen_prompt_lp(position_obj, token_id):
    if position_obj is None:
        return None

    if isinstance(position_obj, dict):
        if token_id in position_obj:
            return _lp_value(position_obj[token_id])
        if str(token_id) in position_obj:
            return _lp_value(position_obj[str(token_id)])

    try:
        return _lp_value(position_obj[token_id])
    except Exception:
        return None

SCORE_PARAMS = SamplingParams(
    temperature=0.0,
    max_tokens=1,
    prompt_logprobs=1,
)

def score_all():
    outputs = llm.generate(
        TOKEN_PROMPTS,
        SCORE_PARAMS,
        use_tqdm=False,
    )

    nlls = []

    for case, out in zip(cases, outputs):
        ids = list(out.prompt_token_ids)
        plps = out.prompt_logprobs

        if plps is None:
            raise RuntimeError("vLLM returned no prompt_logprobs.")

        start = max(1, min(case["answer_start"], len(ids)-1))
        vals = []

        for i in range(start, len(ids)):
            lp = _chosen_prompt_lp(plps[i], int(ids[i]))
            if lp is not None and np.isfinite(lp):
                vals.append(-lp)

        if not vals:
            raise RuntimeError(
                "Could not extract reference-token prompt logprobs. "
                "Stop before the causal sweep."
            )

        nlls.append(float(np.mean(vals)))

    return np.asarray(nlls, dtype=np.float64)

clear_control()
BASE_NLL = score_all()

target_mask = eval_df["kind"].values == "target"
control_mask = eval_df["kind"].values == "control"

print("Baseline target NLL :", BASE_NLL[target_mask].mean())
print("Baseline control NLL:", BASE_NLL[control_mask].mean())

# Phase B — Causal layer sweep

For each sparse layer, all routed-expert weights are set to zero while the
shared-expert path remains untouched.

The score is:

\[
S=\Delta NLL_{target} - \lambda \max(0,\Delta NLL_{control})
\]

Routing frequency is not involved in selection.

In [ ]:
from tqdm.auto import tqdm
import time

CONTROL_PENALTY = 0.75

def summarize(ablated):
    delta = np.asarray(ablated) - BASE_NLL
    td = float(delta[target_mask].mean())
    cd = float(delta[control_mask].mean())

    return {
        "target_delta_nll": td,
        "control_delta_nll": cd,
        "causal_specificity": td - CONTROL_PENALTY * max(cd, 0.0),
        "per_example_delta": delta,
    }

layer_rows = []
t0 = time.time()

for layer_name in tqdm(MOE_LAYERS, desc="39-layer causal sweep"):
    write_control(zero_layers=[layer_name])
    nll = score_all()
    clear_control()

    m = summarize(nll)
    layer_rows.append({
        "layer_name": layer_name,
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
    })

layer_df = pd.DataFrame(layer_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

print(f"Sweep time: {(time.time()-t0)/60:.2f} min")
display(layer_df.head(15))

layer_df.to_csv(WORK / "causal_layer_scores.csv", index=False)

# Phase C — Hierarchical expert group search

Within the strongest layers, search groups instead of brute-forcing 256
individual experts.

Default research settings:

- initial group size = 32;
- beam width = 3;
- randomized expert order;
- fixed-routing ablation;
- descend to individual experts.

In [ ]:
def intervention_score(layer_name, expert_ids, renormalize=False):
    write_control(
        ablate={layer_name: [int(x) for x in expert_ids]},
        renormalize=renormalize,
    )
    nll = score_all()
    clear_control()

    m = summarize(nll)
    return {
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
        "per_example_delta": m["per_example_delta"],
    }

def hierarchical_search(
    layer_name,
    seed=17,
    initial_group_size=32,
    beam_width=3,
):
    rng = np.random.default_rng(seed)
    order = rng.permutation(256).tolist()

    frontier = [
        order[i:i+initial_group_size]
        for i in range(0, 256, initial_group_size)
    ]

    history = []
    level = 0

    while frontier:
        current = []

        for group in tqdm(
            frontier,
            desc=f"{layer_name} level {level}",
            leave=False,
        ):
            m = intervention_score(layer_name, group)

            rec = {
                "layer_name": layer_name,
                "seed": int(seed),
                "level": int(level),
                "group_size": len(group),
                "experts": list(map(int, group)),
                "target_delta_nll": m["target_delta_nll"],
                "control_delta_nll": m["control_delta_nll"],
                "causal_specificity": m["causal_specificity"],
            }

            history.append(rec)
            current.append(rec)

        current.sort(
            key=lambda x: x["causal_specificity"],
            reverse=True,
        )
        keep = current[:beam_width]

        if all(x["group_size"] == 1 for x in keep):
            break

        next_frontier = []

        for x in keep:
            g = x["experts"]
            if len(g) == 1:
                next_frontier.append(g)
            else:
                mid = len(g) // 2
                next_frontier.extend([g[:mid], g[mid:]])

        frontier = [x for x in next_frontier if x]
        level += 1

    hist = pd.DataFrame(history)
    leaf_size = hist["group_size"].min()
    leaves = hist[hist.group_size == leaf_size].sort_values(
        "causal_specificity",
        ascending=False,
    )

    return hist, leaves

## 9 — Search top three causal layers

In [ ]:
TOP_LAYERS = 3
SEARCH_SEED = 17
INITIAL_GROUP_SIZE = 32
BEAM_WIDTH = 3

candidate_layers = layer_df.head(TOP_LAYERS)["layer_name"].tolist()
print("Candidate layers:", candidate_layers)

histories = []
leaves_all = []

for layer_name in candidate_layers:
    hist, leaves = hierarchical_search(
        layer_name,
        seed=SEARCH_SEED,
        initial_group_size=INITIAL_GROUP_SIZE,
        beam_width=BEAM_WIDTH,
    )
    histories.append(hist)
    leaves_all.append(leaves)

group_history = pd.concat(histories, ignore_index=True)
leaf_df = pd.concat(leaves_all, ignore_index=True)

group_history.to_json(
    WORK / "hierarchical_group_history.json",
    orient="records",
    indent=2,
)

display(
    leaf_df[
        [
            "layer_name",
            "experts",
            "target_delta_nll",
            "control_delta_nll",
            "causal_specificity",
        ]
    ].head(20)
)

## 10 — Exact individual expert validation

In [ ]:
leaf_pairs = sorted({
    (str(r.layer_name), int(e))
    for r in leaf_df.itertuples(index=False)
    for e in r.experts
})

print("Leaf candidates:", len(leaf_pairs))

individual_rows = []

for layer_name, expert_id in tqdm(
    leaf_pairs,
    desc="Individual causal validation",
):
    m = intervention_score(layer_name, [expert_id])

    individual_rows.append({
        "layer_name": layer_name,
        "expert": expert_id,
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
        "per_example_delta": m["per_example_delta"].tolist(),
    })

individual_df = pd.DataFrame(individual_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

display(individual_df.head(20))

individual_df.drop(columns=["per_example_delta"]).to_csv(
    WORK / "individual_causal_experts.csv",
    index=False,
)

## 11 — Bootstrap confidence intervals

In [ ]:
def bootstrap_specificity(delta, n_boot=4000, seed=123):
    rng = np.random.default_rng(seed)
    d = np.asarray(delta, dtype=np.float64)

    t = d[target_mask]
    c = d[control_mask]

    vals = np.empty(n_boot, dtype=np.float64)

    for i in range(n_boot):
        tb = rng.choice(t, size=len(t), replace=True).mean()
        cb = rng.choice(c, size=len(c), replace=True).mean()
        vals[i] = tb - CONTROL_PENALTY * max(cb, 0.0)

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

boot_rows = []

for r in individual_df.itertuples(index=False):
    boot_rows.append({
        "layer_name": r.layer_name,
        "expert": int(r.expert),
        **bootstrap_specificity(r.per_example_delta),
    })

boot_df = pd.DataFrame(boot_rows)

final_df = individual_df.merge(
    boot_df,
    on=["layer_name", "expert"],
    how="left",
).sort_values(
    ["p_positive", "causal_specificity"],
    ascending=False,
).reset_index(drop=True)

display(
    final_df[
        [
            "layer_name",
            "expert",
            "target_delta_nll",
            "control_delta_nll",
            "causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "p_positive",
        ]
    ].head(20)
)

## 12 — Renormalization robustness

In [ ]:
ROBUST_TOP_N = min(10, len(final_df))
robust_rows = []

for r in tqdm(
    list(final_df.head(ROBUST_TOP_N).itertuples(index=False)),
    desc="Renormalized re-test",
):
    m = intervention_score(
        r.layer_name,
        [int(r.expert)],
        renormalize=True,
    )

    robust_rows.append({
        "layer_name": r.layer_name,
        "expert": int(r.expert),
        "renorm_target_delta_nll": m["target_delta_nll"],
        "renorm_control_delta_nll": m["control_delta_nll"],
        "renorm_causal_specificity": m["causal_specificity"],
    })

robust_df = pd.DataFrame(robust_rows)

final_robust = final_df.merge(
    robust_df,
    on=["layer_name", "expert"],
    how="left",
)

display(
    final_robust[
        [
            "layer_name",
            "expert",
            "causal_specificity",
            "renorm_causal_specificity",
            "p_positive",
        ]
    ].head(ROBUST_TOP_N)
)

final_robust.drop(columns=["per_example_delta"]).to_csv(
    WORK / "final_causal_candidates.csv",
    index=False,
)

# Phase D — Routing diagnostic AFTER causal selection

Capture routing on the same evaluation prompts with no ablation and compare
routing mass against causal rank.

This is observational evidence only; it is never used to choose the candidates.

In [ ]:
routing_base = WORK / "eval_routing"
remove_capture(routing_base)

write_control(
    capture=True,
    output_base=routing_base,
)

_ = llm.generate(
    TOKEN_PROMPTS,
    SamplingParams(temperature=0.0, max_tokens=1),
    use_tqdm=False,
)

clear_control()

route_records = read_capture(routing_base)

route_rows = []

for rec in route_records:
    tokens = max(1, int(rec["tokens"]))

    for eid_s, count in rec["counts"].items():
        route_rows.append({
            "layer_name": rec["layer_name"],
            "expert": int(eid_s),
            "selected_rate": int(count) / tokens,
            "routing_mass": float(rec["weight_sums"].get(eid_s, 0.0)) / tokens,
        })

routing_df = (
    pd.DataFrame(route_rows)
    .groupby(["layer_name","expert"], as_index=False)
    .agg(
        selected_rate=("selected_rate","mean"),
        routing_mass=("routing_mass","mean"),
    )
)

comparison = final_robust.merge(
    routing_df,
    on=["layer_name","expert"],
    how="left",
).fillna({
    "selected_rate": 0.0,
    "routing_mass": 0.0,
})

comparison["causal_rank"] = comparison["causal_specificity"].rank(
    ascending=False,
    method="min",
)
comparison["routing_rank"] = comparison["routing_mass"].rank(
    ascending=False,
    method="min",
)
comparison["rank_gap"] = (
    comparison["routing_rank"]
    - comparison["causal_rank"]
)

display(
    comparison[
        [
            "layer_name",
            "expert",
            "causal_specificity",
            "routing_mass",
            "causal_rank",
            "routing_rank",
            "rank_gap",
        ]
    ].sort_values("causal_rank").head(20)
)

comparison.drop(columns=["per_example_delta"]).to_csv(
    WORK / "routing_vs_causality.csv",
    index=False,
)

## 13 — Optional same-layer pair / coalition test

In [ ]:
from itertools import combinations

RUN_PAIR_TESTS = True
PAIR_TOP_N = min(8, len(final_robust))

pair_rows = []

if RUN_PAIR_TESTS:
    top = final_robust.head(PAIR_TOP_N)

    for layer_name, group in top.groupby("layer_name"):
        rows = list(group.itertuples(index=False))

        for a, b in combinations(rows, 2):
            m = intervention_score(
                layer_name,
                [int(a.expert), int(b.expert)],
            )

            interaction = (
                m["causal_specificity"]
                - float(a.causal_specificity)
                - float(b.causal_specificity)
            )

            pair_rows.append({
                "layer_name": layer_name,
                "expert_a": int(a.expert),
                "expert_b": int(b.expert),
                "pair_causal_specificity": m["causal_specificity"],
                "interaction_score": interaction,
            })

pair_df = pd.DataFrame(pair_rows)

if not pair_df.empty:
    pair_df = pair_df.sort_values(
        "pair_causal_specificity",
        ascending=False,
    )
    display(pair_df)
    pair_df.to_csv(
        WORK / "expert_pair_interactions.csv",
        index=False,
    )
else:
    print("No same-layer top-candidate pairs to test.")

## 14 — Export manifest + ZIP

In [ ]:
from datetime import datetime, timezone
import shutil, json

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_path": str(MODEL_PATH),
    "backend": "vLLM",
    "vllm_version": vllm.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_vram_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
    "cpu_cores": os.cpu_count(),
    "system_ram_gib": psutil.virtual_memory().total / 2**30,
    "num_sparse_layers_seen": len(MOE_LAYERS),
    "eval_examples": len(eval_df),
    "top_layers_searched": TOP_LAYERS,
    "initial_group_size": INITIAL_GROUP_SIZE,
    "beam_width": BEAM_WIDTH,
    "search_seed": SEARCH_SEED,
    "control_penalty": CONTROL_PENALTY,
    "fixed_routing": True,
}

(WORK / "manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

archive = shutil.make_archive(
    str(WORK),
    "zip",
    root_dir=WORK,
)

print("Results dir:", WORK)
print("ZIP:", archive)

display(
    final_robust[
        [
            "layer_name",
            "expert",
            "target_delta_nll",
            "control_delta_nll",
            "causal_specificity",
            "renorm_causal_specificity",
            "p_positive",
        ]
    ].head(15)
)

# Research-quality run after the notebook works end-to-end

The bundled 24 examples are a **pipeline validation set**, not enough for a
paper claim.

For the real study:

1. 50–200+ target examples.
2. 50–200+ matched control examples.
3. Separate selection and held-out evaluation sets.
4. Repeat hierarchical search with at least two shuffled expert orderings.
5. Compare equal-budget selectors:
   - random;
   - routing frequency;
   - gradient/saliency where available;
   - causal selection.
6. Re-test individual experts with both routing-mass semantics.
7. Test a small number of coalitions.
8. Only then move the winning expert blocks into the **training/surgery stage**.

### Training stage

A 48GB L40S is excellent for the quantized causal atlas. Full-rank surgical
training needs a separate hybrid implementation because vLLM is inference-first
and current Transformers dequantizes Laguna's fused MoE experts.

The next training design should keep the frozen backbone quantized and
materialize only the validated expert blocks as trainable BF16 modules.